In [ ]:
import pandas as pd
import numpy as np
import sys
import matplotlib.pyplot as plt

sys.path.insert(0, '..')
from src.effects import smd

parquet_path = "../data/processed/criteo-uplift-v2.1.parquet"

In [ ]:
df1 = pd.read_parquet(parquet_path)

# Shuffle all rows. Use a smaller frac to work with fewer rows.
df_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
q = np.linspace(0, 1, 1001)

plt.figure(figsize=(10, 5))
for i in range(12):
    feature_quantiles =df_shuffled[f'f{i}'].quantile(q)
    plt.subplot(3,4,i+1)
    plt.plot(q * 100, feature_quantiles.values, linewidth=1, alpha=0.7, label=f'f{i}')
    plt.title(f'f{i}')
plt.suptitle('Feature Quantiles (sorted values)')
plt.xlabel('Percentile of rows (%)')
plt.ylabel('Feature Value')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
treated_positive = df_shuffled[df_shuffled['treatment']==1]['f0'].quantile(q)
untreated_positive = df_shuffled[df_shuffled['treatment']==0]['f0'].quantile(q)
plt.plot(q * 100, treated_positive.values, linewidth=2, color='blue', label='Ad group', alpha=0.9)
plt.plot(q * 100, untreated_positive.values, linewidth=2, color='orange', label='No-ad group', alpha=0.9)
plt.title('Feature 0 Quantiles by Treatment Group')
plt.xlabel('Percentile of rows (%)')
plt.ylabel('Feature 0 Value')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
for i in range(12):
    SMD = smd(df_shuffled, f'f{i}')
    plt.bar(f'f{i}', SMD, color='blue', alpha=0.9)
    print(f'SMD for f{i}: {SMD:.4f}')
plt.axhline(0.1, color='red', linestyle='--', linewidth=1)
plt.axhline(-0.1, color='red', linestyle='--', linewidth=1)
plt.title('SMD by Feature: Ad group vs No-ad group')
plt.xlabel('Feature')
plt.ylabel('SMD Value')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/smd_balance.png', dpi=150)
plt.show()

In [ ]:
exposure_mean_treated = df_shuffled[df_shuffled['treatment']==1]['exposure'].mean()
exposure_mean_untreated = df_shuffled[df_shuffled['treatment']==0]['exposure'].mean()
print(f'Mean exposure for Ad group: {exposure_mean_treated:.6f}')
print(f'Mean exposure for No-ad group: {exposure_mean_untreated:.6f}')

exposure_rate_diff = exposure_mean_treated - exposure_mean_untreated
print(f'Exposure rate difference (share of Ad group that saw an ad; not a treatment effect): {exposure_rate_diff:.6f}')

treatment_share = (df_shuffled['treatment'] == 1).mean()
print(f'Share of Ad group: {treatment_share:.4f}')

exposure_zero_no_ad = df_shuffled[df_shuffled['treatment']==0]['exposure'].value_counts()
print(f'Exposure distribution for No-ad group:\n{exposure_zero_no_ad}')